In [0]:
CATALOG = "cat_medallion"
SCHEMA  = "source_1"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.poc3_customer_source"
SCD1_TABLE   = f"{CATALOG}.{SCHEMA}.poc3_dim_customer_scd1"
SCD2_TABLE   = f"{CATALOG}.{SCHEMA}.poc3_dim_customer_scd2"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_medallion.source_1.poc3_dim_customer_scd2 (
    customer_sk        BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY,
    customer_id        STRING NOT NULL,
    name               STRING,
    email              STRING,
    city               STRING,
    effective_start_date DATE NOT NULL,
    effective_end_date   DATE NOT NULL,
    current_flag       BOOLEAN NOT NULL
)
USING DELTA;

# Insert SCD2 initial data

In [0]:
from pyspark.sql.window import *
from pyspark.sql.functions import *

initial_scd2 = (
    spark.table(SOURCE_TABLE)
    .select("customer_id", "name", "email", "city")
    .filter("batch_seq = 1")
    .orderBy("customer_id")
)

initial_scd2 = (
    initial_scd2
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date",   lit('2999-12-31').cast("date"))
    .withColumn("current_flag",         lit(True))
    .select(
        "customer_id", "name", "email", "city",
        "effective_start_date", "effective_end_date", "current_flag",
    )
)

(
    initial_scd2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .saveAsTable(SCD2_TABLE)
)

display(spark.table(SCD2_TABLE).orderBy("customer_sk"))

In [0]:
%sql
MERGE INTO cat_medallion.source_1.poc3_dim_customer_scd2 AS t
USING (
    SELECT customer_id, name, email, city,ingest_ts
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY customer_id
                   ORDER BY ingest_ts DESC
               ) AS rn
        FROM cat_medallion.source_1.poc3_customer_source
        WHERE batch_seq = 2
    )
    WHERE rn = 1
) AS s
ON t.customer_id = s.customer_id
AND t.current_flag = TRUE

WHEN MATCHED AND (
       NOT (t.name  <=> s.name)
    OR NOT (t.email <=> s.email)
    OR NOT (t.city  <=> s.city)
)
THEN UPDATE SET
    t.effective_end_date = s.ingest_ts,
    t.current_flag = FALSE;

In [0]:
%sql
MERGE INTO cat_medallion.source_1.poc3_dim_customer_scd2 AS t
USING (
    SELECT customer_id, name, email, city,ingest_ts
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY customer_id
                   ORDER BY ingest_ts DESC
               ) AS rn
        FROM cat_medallion.source_1.poc3_customer_source
        WHERE batch_seq = 2
    )
    WHERE rn = 1
) AS s
ON t.customer_id = s.customer_id
AND t.current_flag = TRUE

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    name,
    email,
    city,
    effective_start_date,
    effective_end_date,
    current_flag
)
VALUES (
    s.customer_id,
    s.name,
    s.email,
    s.city,
    s.ingest_ts,
    TIMESTAMP '9999-12-31 23:59:59',
    TRUE
);

In [0]:
%sql
SELECT * FROM cat_medallion.source_1.poc3_dim_customer_scd2